In [1]:
!pip install transformers peft torch --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from typing import Optional, Dict, Tuple
from pathlib import Path
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


In [3]:
GCB_MODEL_NAME  = 'microsoft/graphcodebert-base'
VULB_MODEL_NAME = 'microsoft/codebert-base'

HIDDEN_DIM      = 768
NUM_CWE_CLASSES = 7
GCB_MAX_LEN     = 256
VULB_MAX_LEN    = 256
DROPOUT_RATE    = 0.1

LORA_R       = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
LORA_TARGETS = ['query', 'value']

CWE_LABEL_MAP = {
    'CWE-077': 0, 'CWE-601': 1, 'CWE-022': 2,
    'CWE-094': 3, 'CWE-089': 4, 'CWE-352': 5, 'CWE-079': 6,
}
INV_CWE_MAP = {v: k for k, v in CWE_LABEL_MAP.items()}

print('Config loaded')

Config loaded


In [4]:
class FusionProjectionLayer(nn.Module):
    """
    Concat + Project Fusion.

    Input :
        gcb_cls  : (batch, 768) — CLS من GraphCodeBERT
        vulb_cls : (batch, 768) — CLS من VulBERTa

    Steps:
        1. Concat  → (batch, 1536)
        2. Linear 1536 → 768
        3. LayerNorm
        4. GELU
        5. Dropout

    Output:
        fused : (batch, 768)  ← single unified representation
    """

    def __init__(
        self,
        hidden_dim: int   = HIDDEN_DIM,
        proj_dim:   int   = HIDDEN_DIM,
        dropout:    float = DROPOUT_RATE,
    ):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, proj_dim),  # 1536 → 768
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Xavier initialization
        nn.init.xavier_uniform_(self.projection[0].weight)
        nn.init.zeros_(self.projection[0].bias)

    def forward(
        self,
        gcb_cls:  torch.Tensor,  # (B, 768)  GraphCodeBERT
        vulb_cls: torch.Tensor,  # (B, 768)  VulBERTa
    ) -> torch.Tensor:           # (B, 768) single output

        # Concat: (B,768) + (B,768) → (B,1536)
        concat = torch.cat([gcb_cls, vulb_cls], dim=-1)

        # Project: (B,1536) → (B,768)
        return self.projection(concat)


# Sanity check
_fl  = FusionProjectionLayer()
_g   = torch.randn(4, 768)
_v   = torch.randn(4, 768)
_out = _fl(_g, _v)
print(f'gcb_cls  : {tuple(_g.shape)}')
print(f'vulb_cls : {tuple(_v.shape)}')
print(f'concat   : {tuple(torch.cat([_g,_v],dim=-1).shape)}')
print(f'fused    : {tuple(_out.shape)}  <- single output')
del _fl, _g, _v, _out

gcb_cls  : (4, 768)
vulb_cls : (4, 768)
concat   : (4, 1536)
fused    : (4, 768)  <- single output


In [5]:
class DualTokenizerDataset(Dataset):
    """
    get sample:
        gcb_input_ids, gcb_attention_mask   (من GraphCodeBERT tokenizer)
        vulb_input_ids, vulb_attention_mask (من VulBERTa tokenizer)
        label
    """

    def __init__(self, records, gcb_tokenizer, vulb_tokenizer,
                 gcb_max_len=GCB_MAX_LEN, vulb_max_len=VULB_MAX_LEN):
        self.records      = records
        self.gcb_tok      = gcb_tokenizer
        self.vulb_tok     = vulb_tokenizer
        self.gcb_max_len  = gcb_max_len
        self.vulb_max_len = vulb_max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r     = self.records[idx]
        code  = ' '.join(r['lines'])
        label = CWE_LABEL_MAP.get(r.get('cwe_id', ''), -1)

        # Tokenize with GraphCodeBERT tokenizer
        gcb = self.gcb_tok(
            code, max_length=self.gcb_max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        # Tokenize with VulBERTa tokenizer (SEPARATE!)
        vulb = self.vulb_tok(
            code, max_length=self.vulb_max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        return {
            'gcb_input_ids':       gcb['input_ids'].squeeze(0),
            'gcb_attention_mask':  gcb['attention_mask'].squeeze(0),
            'vulb_input_ids':      vulb['input_ids'].squeeze(0),
            'vulb_attention_mask': vulb['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
        }

print('DualTokenizerDataset defined')

DualTokenizerDataset defined


In [6]:
class VulBERTaFusionModel(nn.Module):
    """
    Dual-encoder fusion model.

    forward() inputs:
        gcb_input_ids, gcb_attention_mask   ← from GraphCodeBERT tokenizer
        vulb_input_ids, vulb_attention_mask ← from VulBERTa tokenizer

    forward() output:
        logits     : (B, 7)   — CWE class scores
        fused_repr : (B, 768) — the single fused representation
        loss       : scalar   —  labels 
    """

    def __init__(
        self,
        gcb_model_name:  str   = GCB_MODEL_NAME,
        vulb_model_name: str   = VULB_MODEL_NAME,
        num_labels:      int   = NUM_CWE_CLASSES,
        hidden_dim:      int   = HIDDEN_DIM,
        dropout:         float = DROPOUT_RATE,
        freeze_vulberta: bool  = True,
    ):
        super().__init__()

        # ── Encoder 1: GraphCodeBERT + LoRA ───────────────────
        print(f'Loading GraphCodeBERT ...')
        _gcb  = AutoModel.from_pretrained(gcb_model_name)
        _lora = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS, bias='none',
        )
        self.graphcodebert = get_peft_model(_gcb, _lora)
        self.graphcodebert.print_trainable_parameters()

        # ── Encoder 2: VulBERTa ───────────────────────────────
        print(f'Loading VulBERTa encoder ...')
        self.vulberta = AutoModel.from_pretrained(vulb_model_name)
        if freeze_vulberta:
            for p in self.vulberta.parameters():
                p.requires_grad = False
            print('VulBERTa FROZEN')

        # ── Fusion Layer  ──────────────────────
        self.fusion = FusionProjectionLayer(
            hidden_dim=hidden_dim, proj_dim=hidden_dim, dropout=dropout
        )

        # ── CWE Classification Head ────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_labels),
        )

        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-1)

    @staticmethod
    def _get_cls(encoder, input_ids, attention_mask):
        """بيشغل الـ encoder ويرجع [CLS] token."""
        return encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]  # position 0 = [CLS]

    def forward(
        self,
        gcb_input_ids:       torch.Tensor,  # (B, GCB_MAX_LEN)
        gcb_attention_mask:  torch.Tensor,  # (B, GCB_MAX_LEN)
        vulb_input_ids:      torch.Tensor,  # (B, VULB_MAX_LEN)
        vulb_attention_mask: torch.Tensor,  # (B, VULB_MAX_LEN)
        labels: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:

        # Step 1: encoder for each token
        gcb_cls  = self._get_cls(
            self.graphcodebert, gcb_input_ids, gcb_attention_mask
        )  # (B, 768)

        vulb_cls = self._get_cls(
            self.vulberta, vulb_input_ids, vulb_attention_mask
        )  # (B, 768)

        # Step 2: Fusion → single representation
        fused = self.fusion(gcb_cls, vulb_cls)   # (B, 768)

        # Step 3: CWE classification
        logits = self.classifier(fused)          # (B, 7)

        out = {'logits': logits, 'fused_repr': fused}
        if labels is not None:
            out['loss'] = self.loss_fn(logits, labels)
        return out


print('VulBERTaFusionModel defined')

VulBERTaFusionModel defined


#### **Build Model & Inspect**

In [7]:
model = VulBERTaFusionModel(freeze_vulberta=True).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal     : {total:,}')
print(f'Trainable : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'\nFusion layer:')
print(model.fusion)

Loading GraphCodeBERT ...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hende\.cache\huggingface\hub\models--microsoft--graphcodebert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hende\.cache\huggingface\hub\models--microsoft--codebert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

VulBERTa FROZEN

Total     : 251,066,119
Trainable : 1,774,855 (0.71%)

Fusion layer:
FusionProjectionLayer(
  (projection): Sequential(
    (0): Linear(in_features=1536, out_features=768, bias=True)
    (1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
  )
)


#### **Forward Pass Test — 4 Separate Inputs**

In [8]:
gcb_tokenizer  = AutoTokenizer.from_pretrained(GCB_MODEL_NAME)
vulb_tokenizer = AutoTokenizer.from_pretrained(VULB_MODEL_NAME)

codes = [
    "query = 'SELECT * FROM users WHERE id=' + user_input",
    "subprocess.run(cmd, shell=True)",
    "return redirect(request.args.get('next', '/'))",
]

# Tokenize SEPARATELY
gcb_enc  = gcb_tokenizer(codes,  max_length=GCB_MAX_LEN,  padding='max_length', truncation=True, return_tensors='pt').to(device)
vulb_enc = vulb_tokenizer(codes, max_length=VULB_MAX_LEN, padding='max_length', truncation=True, return_tensors='pt').to(device)
labels   = torch.tensor([4, 3, 1], dtype=torch.long).to(device)

model.eval()
with torch.no_grad():
    out = model(
        gcb_input_ids       = gcb_enc['input_ids'],
        gcb_attention_mask  = gcb_enc['attention_mask'],
        vulb_input_ids      = vulb_enc['input_ids'],
        vulb_attention_mask = vulb_enc['attention_mask'],
        labels = labels,
    )

print('Forward pass OK!')
print(f'  gcb tokens  : {gcb_enc["input_ids"].shape}')
print(f'  vulb tokens : {vulb_enc["input_ids"].shape}')
print(f'  fused_repr  : {out["fused_repr"].shape}  <- single output')
print(f'  logits      : {out["logits"].shape}')
print(f'  loss        : {out["loss"].item():.4f}')

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Forward pass OK!
  gcb tokens  : torch.Size([3, 256])
  vulb tokens : torch.Size([3, 256])
  fused_repr  : torch.Size([3, 768])  <- single output
  logits      : torch.Size([3, 7])
  loss        : 1.9589


#### **Unit Tests**

In [9]:
def test_fusion_layer():
    fl = FusionProjectionLayer(dropout=0.0)
    fl.eval()
    print('Running tests...')

    # T1: shape
    o = fl(torch.randn(8, 768), torch.randn(8, 768))
    assert o.shape == (8, 768)
    print('  T1 PASSED: output shape (8, 768)')

    # T2: single Tensor output (not tuple)
    assert isinstance(o, torch.Tensor)
    print('  T2 PASSED: output is a single Tensor')

    # T3: concat dim is 1536
    g, v = torch.randn(1,768), torch.randn(1,768)
    assert torch.cat([g,v], dim=-1).shape[-1] == 1536
    print('  T3 PASSED: concat produces 1536 before projection')

    # T4: gradients flow through both paths
    fl.train()
    g2 = torch.randn(2, 768, requires_grad=True)
    v2 = torch.randn(2, 768, requires_grad=True)
    fl(g2, v2).sum().backward()
    assert g2.grad is not None and v2.grad is not None
    print('  T4 PASSED: gradients flow through both encoder paths')

    # T5: different inputs → different outputs
    fl.eval()
    oa = fl(torch.randn(2,768), torch.randn(2,768))
    ob = fl(torch.randn(2,768), torch.randn(2,768))
    assert not torch.allclose(oa, ob)
    print('  T5 PASSED: different inputs → different fused outputs')

    print('\nAll tests passed!')

test_fusion_layer()

Running tests...
  T1 PASSED: output shape (8, 768)
  T2 PASSED: output is a single Tensor
  T3 PASSED: concat produces 1536 before projection
  T4 PASSED: gradients flow through both encoder paths
  T5 PASSED: different inputs → different fused outputs

All tests passed!


#### **DataLoader Helper**

In [10]:
def build_dataloaders(unified_path, gcb_tokenizer, vulb_tokenizer, batch_size=16):
    path = Path(unified_path)
    if not path.exists():
        print(f'File not found: {unified_path}')
        return None, None

    records = [json.loads(l) for l in open(path)]
    train   = [r for r in records if r['split_origin'] == 'train']
    val     = [r for r in records if r['split_origin'] == 'val']

    train_loader = DataLoader(DualTokenizerDataset(train, gcb_tokenizer, vulb_tokenizer), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(DualTokenizerDataset(val,   gcb_tokenizer, vulb_tokenizer), batch_size=batch_size, shuffle=False)

    print(f'Train: {len(train):,} samples | Val: {len(val):,} samples')
    return train_loader, val_loader


# Uncomment after running previous notebooks:
# train_loader, val_loader = build_dataloaders('datasets/UNIFIED.jsonl', gcb_tokenizer, vulb_tokenizer)
print('build_dataloaders() ready')

build_dataloaders() ready


In [11]:
# save
import os

IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    SAVE_DIR = Path('/content/drive/MyDrive/CSI_Project/checkpoints')
else:
    SAVE_DIR = Path('./checkpoints')

SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = SAVE_DIR / 'vulberta_fusion_init.pt'

torch.save(model.state_dict(), SAVE_PATH)
print(f'Saved: {SAVE_PATH} ({os.path.getsize(SAVE_PATH)/(1024**2):.1f} MB)')

Saved: checkpoints\vulberta_fusion_init.pt (958.0 MB)
